# 🗣️ Resonova — Kaggle Session Template
## Phase 0: Backup Compute Environment

**Purpose:** Backup session-starter for Resonova when Colab GPU is unavailable or quota is exhausted.
Especially useful for Phase 3 ablation runs (30 GPU-hr/week budget is more predictable than Colab).

---
### ⚡ Before running this notebook:
1. **Settings → Accelerator → GPU T4 x2** (or P100 — both work)
2. **Settings → Internet → On** (required for pip installs and GitHub)
3. This notebook writes outputs to `/kaggle/working/` — **download important files before session ends**

---
### ⚠️ Key differences vs Colab:
| | Google Colab | Kaggle |
|---|---|---|
| GPU budget | ~12 hr/session (no weekly limit) | 30 GPU-hr/week, 12 hr/session |
| Persistent storage | Google Drive (via mount) | `/kaggle/working/` (download manually) |
| Session recovery | Re-run from Drive | Re-run from `/kaggle/working/` or GitHub |
| Idle disconnect | ~30-60 min idle | ~60 min idle |

**Budget tracking:** Kaggle's 30 GPU-hr/week resets on Saturday. Track usage carefully during Phase 3 ablation.

## STEP 1 — GPU Verification

In [ ]:
# Each cell is self-contained — imports are repeated where needed
import subprocess
import sys
from datetime import datetime

ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('=' * 60)
print('RESONOVA KAGGLE SESSION START — ' + ts)
print('=' * 60)

# --- nvidia-smi ---
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('❌ No GPU detected!')
    print('   Fix: Settings → Accelerator → GPU T4 x2 or P100')
    sys.exit('GPU required. Session aborted.')

print('✅ nvidia-smi: GPU present')
for line in result.stdout.split('\n'):
    if any(g in line for g in ['Tesla', 'T4', 'P100', 'A100', 'V100']):
        print(f'   {line.strip()}')

# --- PyTorch CUDA ---
try:
    import torch
    print(f'\n   PyTorch version : {torch.__version__}')
    print(f'   CUDA available  : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        gpu_name   = torch.cuda.get_device_name(0)
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'   GPU name        : {gpu_name}')
        print(f'   Total VRAM      : {total_vram:.1f} GB')
        if total_vram < 12:
            print('   ⚠️  <12 GB VRAM — use load/unload strategy between pipeline stages')
        else:
            print('   ✅ VRAM sufficient for all Resonova models')
except ImportError:
    print('   ⚠️  PyTorch not yet installed — installing in STEP 5')

print('\n' + '=' * 60)

## STEP 2 — Kaggle Output Paths
Kaggle does NOT have Google Drive. Outputs go to `/kaggle/working/`.  
**Download important files before your session ends** — they do persist between sessions in Kaggle's storage, but are not as reliable as Drive. For critical outputs, download manually.

In [ ]:
import os

# Kaggle working directory — persists in Kaggle's storage
KAGGLE_WORKING  = '/kaggle/working'
OUTPUTS_DIR     = KAGGLE_WORKING + '/resonova_outputs'
CHECKPOINTS_DIR = KAGGLE_WORKING + '/resonova_checkpoints'

for d in [OUTPUTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'  ✅ Output path ready: {d}')

# Set environment variables for Resonova modules to use
os.environ['RESONOVA_OUTPUTS']     = OUTPUTS_DIR
os.environ['RESONOVA_CHECKPOINTS'] = CHECKPOINTS_DIR

print(f'\n📁 CHECKPOINT: All outputs go to {OUTPUTS_DIR}')
print('   Download from Kaggle UI → Output tab before session ends.')
print('   For critical files, also copy to a mounted dataset if needed.')

# ─── GPU-hour budget tracking ─────────────────────────────────────────────
print('\n⏱️  KAGGLE GPU BUDGET REMINDER:')
print('   Free tier: 30 GPU-hr/week, resets Saturday UTC.')
print('   A full Phase 1 run (4 models, 30s clip) ≈ 1.5-2 GPU-hours.')
print('   Phase 3 ablation (2x full pipeline x 3 clips) ≈ 6-10 GPU-hours.')
print('   Check remaining quota: kaggle.com/account → GPU quota')

## STEP 3 — Clone / Pull GitHub Repository

In [ ]:
import os
import subprocess

# ─── UPDATE THIS ──────────────────────────────────────────────────────────
GITHUB_REPO_URL = 'https://github.com/SAK_SHI14/resonova.git'  # ← your GitHub URL
REPO_DIR        = '/kaggle/working/resonova'
# ─────────────────────────────────────────────────────────────────────────

if os.path.isdir(REPO_DIR):
    print(f'Repository present at {REPO_DIR}. Pulling latest...')
    result = subprocess.run(['git', 'pull'], cwd=REPO_DIR, capture_output=True, text=True)
    print(result.stdout.strip() or '  (already up to date)')
else:
    print(f'Cloning {GITHUB_REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', GITHUB_REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'  ✅ Cloned to {REPO_DIR}')
    else:
        raise RuntimeError('Git clone failed: ' + result.stderr)

commit = subprocess.run(
    ['git', 'log', '--oneline', '-1'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(f'  HEAD: {commit.stdout.strip()}')

## STEP 4 — System Dependencies

In [ ]:
# FIX: subprocess imported here — each cell is self-contained
import subprocess
import os

print('Installing system dependencies...')
# Kaggle images already have many system packages; install only what may be missing
os.system('apt-get update -qq')
os.system('apt-get install -y -qq ffmpeg cmake build-essential libsndfile1')

result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    first_line = result.stdout.split('\n')[0]
    print(f'  ✅ ffmpeg: {first_line}')
else:
    print('  ❌ ffmpeg install failed')

## STEP 5 — Install PyTorch (Pinned — First!)

In [ ]:
# FIX: sys imported here for sys.modules reload
import sys
import subprocess

print('Installing pinned PyTorch 1.12.1 + CUDA 11.3...')

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==1.12.1+cu113',
        'torchvision==0.13.1+cu113',
        'torchaudio==0.12.1+cu113',
        '--extra-index-url', 'https://download.pytorch.org/whl/cu113',
    ],
    check=True
)

# Reload torch modules
for mod in list(sys.modules.keys()):
    if mod.startswith('torch'):
        del sys.modules[mod]

import torch
print(f'\n  ✅ torch=={torch.__version__}, CUDA={torch.cuda.is_available()}')
assert '1.12' in torch.__version__, f'Expected 1.12.x, got {torch.__version__}'

## STEP 6 — Install Requirements + Resonova

> ℹ️ Uses `subprocess` instead of `!` shell magic so Python variables can be passed safely.

In [ ]:
# FIX: all imports self-contained; use subprocess for variable paths; sys imported here
import os
import sys
import subprocess

REPO_DIR = '/kaggle/working/resonova'

print('Installing requirements.txt...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', os.path.join(REPO_DIR, 'requirements.txt')],
    check=True
)

# Protect numpy version
import numpy as np
if np.__version__ != '1.23.5':
    print(f'  ⚠️  numpy upgraded to {np.__version__} — downgrading...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.23.5'],
        check=True
    )
    print('  ✅ numpy downgraded to 1.23.5')
else:
    print(f'  ✅ numpy=={np.__version__}')

# Install Resonova in editable mode
print('\nInstalling Resonova in editable mode...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR],
    check=True
)

sys.path.insert(0, REPO_DIR)
import resonova as _resonova
print(f'  ✅ resonova=={_resonova.__version__} importable')

## STEP 7 — Session Environment Variables

In [ ]:
import os

os.environ['RESONOVA_OUTPUTS']     = '/kaggle/working/resonova_outputs'
os.environ['RESONOVA_CHECKPOINTS'] = '/kaggle/working/resonova_checkpoints'

WAV2LIP_REPO_PATH       = '/kaggle/working/Wav2Lip'
WAV2LIP_CHECKPOINT_PATH = '/kaggle/working/Wav2Lip/checkpoints/wav2lip_gan.pth'

if os.path.isdir(WAV2LIP_REPO_PATH):
    os.environ['WAV2LIP_REPO_PATH']       = WAV2LIP_REPO_PATH
    os.environ['WAV2LIP_CHECKPOINT_PATH'] = WAV2LIP_CHECKPOINT_PATH
    print('  ✅ Wav2Lip repo: ' + WAV2LIP_REPO_PATH)
    ckpt_exists = os.path.isfile(WAV2LIP_CHECKPOINT_PATH)
    # FIX: avoid nested quotes in f-string — use plain variables
    ckpt_icon = '✅' if ckpt_exists else '⚠️ '
    ckpt_msg  = 'found' if ckpt_exists else 'NOT FOUND — run phase1_setup.ipynb STEP 8'
    print(f'  {ckpt_icon} Wav2Lip checkpoint: {ckpt_msg}')
else:
    print('  ℹ️  Wav2Lip not yet cloned — run phase1_setup.ipynb STEP 8')

print('\nEnvironment ready:')
for k in ['RESONOVA_OUTPUTS', 'RESONOVA_CHECKPOINTS', 'WAV2LIP_REPO_PATH', 'WAV2LIP_CHECKPOINT_PATH']:
    print(f'  {k} = {os.environ.get(k, "(not set)")}')

## STEP 8 — Full Environment Audit

In [ ]:
# FIX: sys imported here — this cell was missing the import
import importlib
import sys
import platform
from datetime import datetime

CRITICAL_PACKAGES = [
    ('torch',          'torch'),
    ('torchvision',    'torchvision'),
    ('numpy',          'numpy'),
    ('opencv-python',  'cv2'),
    ('librosa',        'librosa'),
    ('face-alignment', 'face_alignment'),
    ('soundfile',      'soundfile'),
    ('pydub',          'pydub'),
    ('transformers',   'transformers'),
    ('sentencepiece',  'sentencepiece'),
    ('TTS',            'TTS'),
    ('Pillow',         'PIL'),
    ('gradio',         'gradio'),
    ('tqdm',           'tqdm'),
]

ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('=' * 65)
print('RESONOVA KAGGLE AUDIT — ' + ts)
print('=' * 65)
print('Python  : ' + sys.version.split()[0])
print('Platform: ' + platform.platform())
print()

all_ok = True
for pkg_name, import_name in CRITICAL_PACKAGES:
    try:
        mod     = importlib.import_module(import_name)
        version = getattr(mod, '__version__', 'installed')
        print(f'  {pkg_name:<25} {version}')
    except ImportError:
        print(f'  {pkg_name:<25} NOT INSTALLED')
        all_ok = False

print()
print('=' * 65)
status = 'All packages present' if all_ok else 'Some packages missing — run phase1_setup.ipynb'
print(status)
print('=' * 65)

## STEP 9 — Session Ready ✅

### ⚠️ Kaggle session-end checklist
Before your session ends (Kaggle will notify you):
1. **Save outputs**: Important files are in `/kaggle/working/resonova_outputs/` — download via the Output tab
2. **Push code to GitHub**: run the cell below
3. **Record GPU usage**: Note how many GPU hours this session used. Kaggle budgets 30 hr/week.

### GPU hour tracking
| Phase | Expected usage |
|---|---|
| Phase 0 (template verification) | <0.1 hr |
| Phase 1 (4 model tests, 30s clip) | 1.5-2 hr |
| Phase 2 (full pipeline, 3 clips) | 3-5 hr |
| Phase 3 ablation (2x pipeline x 3 clips) | 6-10 hr |
| **Total estimated** | **~15-17 hr** (well within 30 hr/week) |

In [ ]:
# FIX: torch and os imported here — self-contained
import os

try:
    import torch
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
except ImportError:
    gpu_info = 'torch not installed'

print('Resonova Kaggle session is ready.')
print('  Repo   : /kaggle/working/resonova')
print('  Outputs: ' + os.environ.get('RESONOVA_OUTPUTS', 'NOT SET — re-run STEP 2'))
print('  GPU    : ' + gpu_info)
print()
print('Next: open phase1_setup.ipynb to install and test each model.')
print('Remember: download outputs before session ends!')

In [ ]:
# Run this at END of session to push any code changes back to GitHub
import subprocess
import sys

REPO_DIR   = '/kaggle/working/resonova'
COMMIT_MSG = 'Session update'  # ← change this before running

def _run(cmd, cwd=REPO_DIR):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

_run(['git', 'add', '-A'])
rc = _run(['git', 'commit', '-m', COMMIT_MSG])
if rc == 0:
    _run(['git', 'push'])
    print('Code pushed to GitHub.')
else:
    print('Nothing to commit (or commit failed — check output above).')